# LeetCode #1230: Toss Strange Coins

https://leetcode.com/problems/toss-strange-coins/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(2^n)$ | $O(2^n)$ |
| **Optimal: 1D Probability DP ★** | $O(n \cdot \text{target})$ | $O(\text{target})$ |

---

## Understanding the Methods

### Brute Force
Enumerate all $2^n$ subsets of coin outcomes, sum the probabilities of subsets where exactly `target` coins land heads. Exponential — impractical beyond $n \approx 20$.

### Optimal: 1D Probability DP ★
`dp[j]` = probability that exactly `j` coins are heads after processing the first `i` coins. For each coin with probability `p`, update right-to-left: `dp[j] = dp[j-1]*p + dp[j]*(1-p)`. A single array suffices because we traverse in reverse, avoiding overwrite issues.

**Constraints:**
* `1 <= prob.length <= 1000`
* `0 <= prob[i] <= 1`
* `0 <= target <= prob.length`

## Solutions

### C#

In [ ]:
public class Solution {
    public double ProbabilityOfHeads(double[] prob, int target) {
        // dp[j] = probability of exactly j heads so far
        double[] dp = new double[target + 1];
        dp[0] = 1.0;

        foreach (double p in prob) {
            // Traverse right-to-left to avoid using the updated value within this round
            for (int j = Math.Min(target, prob.Length); j >= 1; j--) {
                // j heads: either j-1 heads before this toss (heads) or j heads (tails)
                dp[j] = dp[j - 1] * p + dp[j] * (1 - p);
            }
            // Zero heads: all previous tosses also tails
            dp[0] *= (1 - p);
        }

        return dp[target];
    }
}

### Python

In [ ]:
class Solution:
    def probabilityOfHeads(self, prob: list[float], target: int) -> float:
        # dp[j] = probability of exactly j heads so far
        dp = [0.0] * (target + 1)
        dp[0] = 1.0

        for p in prob:
            # Traverse right-to-left to avoid updating dp[j-1] before using it
            for j in range(min(target, len(prob)), 0, -1):
                # j heads: came from j-1 heads (this toss = heads) OR j heads (this toss = tails)
                dp[j] = dp[j - 1] * p + dp[j] * (1 - p)
            dp[0] *= (1 - p)

        return dp[target]

### Go

In [ ]:
func probabilityOfHeads(prob []float64, target int) float64 {
    // dp[j] = probability of exactly j heads processed so far
    dp := make([]float64, target+1)
    dp[0] = 1.0

    n := len(prob)
    for _, p := range prob {
        // Right-to-left avoids reading already-updated dp[j-1]
        limit := target
        if n < limit { limit = n }
        for j := limit; j >= 1; j-- {
            // j heads: prior j-1 heads then heads, OR prior j heads then tails
            dp[j] = dp[j-1]*p + dp[j]*(1-p)
        }
        dp[0] *= (1 - p)
    }
    return dp[target]
}

### Rust

In [ ]:
impl Solution {
    pub fn probability_of_heads(prob: Vec<f64>, target: i32) -> f64 {
        let t = target as usize;
        // dp[j] = probability of exactly j heads so far
        let mut dp = vec![0.0f64; t + 1];
        dp[0] = 1.0;

        for p in &prob {
            // Traverse right-to-left so dp[j-1] is still the previous round's value
            for j in (1..=t).rev() {
                // j heads: came from j-1 heads (this = heads) or j heads (this = tails)
                dp[j] = dp[j - 1] * p + dp[j] * (1.0 - p);
            }
            dp[0] *= 1.0 - p;
        }
        dp[t]
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `prob = [0.4], target = 1`
`dp[0]=1.0` initially. After coin: `dp[1] = dp[0]*0.4 = 0.4`; `dp[0] = 1.0*0.6 = 0.6`. Answer: **0.4**.

### 2. Slightly Complex
**Input:** `prob = [0.5, 0.5], target = 1`
After coin 1: `dp[1]=0.5, dp[0]=0.5`. After coin 2: `dp[1] = 0.5*0.5 + 0.5*0.5 = 0.5`. Answer: **0.5**, matching binomial intuition.

### 3. Edge Case: Time Factor
**Input:** `prob` has 1000 coins, `target = 500`.
The inner loop runs `target` iterations for each of the 1000 coins, totalling $1000 \times 500 = 500{,}000$ operations — well within $O(n \cdot t)$.

### 4. Edge Case: Space Factor
**Input:** `prob = [0.3]*1000, target = 0`.
Only `dp[0]` is needed; the array has just 1 element, constant space $O(1)$ effectively.

### 5. Almost-Impossible but Plausible
**Input:** `prob = [1.0]*1000, target = 999`.
Every coin must land heads, so probability is 0 (exactly 1000 heads, not 999). `dp[999]` remains 0 because `dp[1000]` would be 1 but is outside our truncated array.